# Protocol 2 - Prediction tasks / levels: residue, domain, protein

Every CPP workflow operates at one of **three prediction levels** - **residue**, **domain**, or **protein** - encoded in the `load_dataset` name prefix (`AA_*`, `DOM_*`, `SEQ_*`). The level is *not* a different algorithm: all three call the same `CPP.run`. What changes is the **unit of comparison** (a window, a part set, or the whole chain) and therefore **how you build `df_parts`**. Getting `df_parts` right is the single most common bottleneck for new users, so this protocol shows the correct construction side-by-side for all three levels.

This protocol is part of the [Protocols catalog](https://github.com/breimanntools/aaanalysis/issues/35). It follows the standard pipeline-chained structure: *When to use it -> Input -> Run -> Output -> How to interpret -> Common mistakes -> Next step.*

## When to use it

Use this protocol when you have a task and are unsure whether it is **residue**, **domain**, or **protein** level - or when you are building `df_parts` for the first time and want a correct template per level.

The level follows the biological question:

- **Residue level** (`AA_*`): *"What distinguishes a position from its neighbours?"* The unit of comparison is a fixed-length **window** anchored at a residue or a scissile bond (e.g. protease cleavage, PTM sites).
- **Domain level** (`DOM_*`): *"What distinguishes one sub-region of a protein from the same region in another protein?"* The unit is a **part** set derived from `tmd_start`/`tmd_stop` (e.g. gamma-secretase substrates compared over their transmembrane domain).
- **Protein level** (`SEQ_*`): *"What distinguishes one whole chain from another?"* The unit is the **whole sequence** as a single part (e.g. aggregation-prone vs. soluble peptides).

## Input

A `df_seq` whose format matches the level (see `SequenceFeature.get_df_parts`), plus a binary `label` column (`1` = test class, `0` = reference class). `load_dataset(name=..., n=N)` returns `2N` rows (N per class); read class sizes from the `label` column, never from `len(df_seq)`.

| Level | Dataset | `df_seq` format | Biological contrast |
| --- | --- | --- | --- |
| residue | `AA_CASPASE3` | **sequence-based** (9-residue windows around the scissile bond, even window = between-residues) | cleaved (`1`) vs. not cleaved (`0`) |
| domain | `DOM_GSEC` | **position-based** (`tmd_start` / `tmd_stop`, 1-based inclusive) | gamma-secretase substrate (`1`) vs. non-substrate (`0`) |
| protein | `SEQ_AMYLO` | **sequence-based** (short full peptides) | aggregation-prone (`1`) vs. soluble (`0`) |

Each upstream protocol feeds this one: residue-level windows come from the *Construct sets / sampling* protocol; domain- and protein-level tables come from `load_dataset` or your own annotation.

In [1]:
import aaanalysis as aa

aa.options["verbose"] = False
aa.options["random_state"] = 42

sf = aa.SequenceFeature()

## Run

The pipeline is identical at every level - `get_df_parts` -> `CPP.run` -> `TreeModel` ranking - and only `df_parts` (and the matching `split_kws`) differ. We build the three `df_parts` first, then run CPP on each.

### Residue level: a window as a single part

The 9-residue window is the unit of comparison, so we map the **whole window onto one part**. We set `jmd_n_len=0` / `jmd_c_len=0` so no flanking is split off and select the neutral single part `tmd` (first-class custom part names such as the Schechter-Berger `P2 P1 | P1' P2'` convention are tracked by [#27](https://github.com/breimanntools/aaanalysis/issues/27)).

In [2]:
df_res = aa.load_dataset(name="AA_CASPASE3", n=5)
labels_res = df_res["label"].to_list()

df_parts_res = sf.get_df_parts(df_seq=df_res, list_parts=["tmd"], jmd_n_len=0, jmd_c_len=0)
df_parts_res.head()

,tmd
entry,
CASPASE3_1_pos4,MSLFDLFRG
CASPASE3_1_pos5,SLFDLFRGF
CASPASE3_1_pos6,LFDLFRGFF
CASPASE3_1_pos7,FDLFRGFFG
CASPASE3_1_pos8,DLFRGFFGF


### Domain level: named parts from `tmd_start` / `tmd_stop`

The position-based format lets `get_df_parts` slice each sequence into the **named parts** `jmd_n` / `tmd` / `jmd_c` (here the TMD *is* the domain of interest). `jmd_n_len` / `jmd_c_len` set how many flanking residues frame the domain.

In [3]:
df_dom = aa.load_dataset(name="DOM_GSEC", n=5)
labels_dom = df_dom["label"].to_list()

df_parts_dom = sf.get_df_parts(df_seq=df_dom, list_parts=["jmd_n", "tmd", "jmd_c"],
                               jmd_n_len=10, jmd_c_len=10)
df_parts_dom.head()

,jmd_n,tmd,jmd_c
entry,,,
Q14802,NSPFYYDWHS,LQVGGLICAGVLCAMGIIIVMSA,KCKCKFGQKS
Q86UE4,LGLEPKRYPG,WVILVGTGALGLLLLFLLGYGWA,AACAGARKKR
Q969W9,FQSMEITELE,FVQIIIIVVVMMVMVVVITCLLS,HYKLSARSFI
P53801,RWGVCWVNFE,ALIITMSVVGGTLLLGIAICCCC,CCRRKRSRKP
Q8IUW5,NDTGNGHPEY,IAYALVPVFFIMGLFGVLICHLL,KKKGYRCTTE


### Protein level: the whole chain as a single part

For whole-chain prediction the entire sequence is the part. As at the residue level we set `jmd_n_len=0` / `jmd_c_len=0` and select the single neutral part `tmd` - but here it spans the full peptide, not a window.

In [4]:
df_prot = aa.load_dataset(name="SEQ_AMYLO", n=5)
labels_prot = df_prot["label"].to_list()

df_parts_prot = sf.get_df_parts(df_seq=df_prot, list_parts=["tmd"], jmd_n_len=0, jmd_c_len=0)
df_parts_prot.head()

,tmd
entry,
AMYLO_1,AAAQAA
AMYLO_2,QSSYSS
AMYLO_3,QSYGQQ
AMYLO_4,QSYNPP
AMYLO_5,QSYSGY


### Run CPP per level

`split_kws` must fit the part lengths (segments/patterns cannot exceed the shortest part). For the short residue window we use small **positional** splits; for the domain we size splits to the 10-residue JMDs; for the protein we use a single **compositional** `Segment(1,1)` whole-part average (position-agnostic, composition-like).

In [5]:
# Residue level: positional splits over the short window
split_kws_res = sf.get_split_kws(split_types=["Segment", "Pattern"], n_split_min=1,
                                 n_split_max=4, steps_pattern=[1, 2], n_min=2, n_max=3, len_max=9)
cpp_res = aa.CPP(df_parts=df_parts_res, split_kws=split_kws_res)
df_feat_res = cpp_res.run(labels=labels_res, n_filter=20, n_jobs=1)  # n_jobs=1 avoids a spawn issue on Python 3.14/macOS

X_res = sf.feature_matrix(features=df_feat_res["feature"], df_parts=df_parts_res)
tm_res = aa.TreeModel().fit(X_res, labels=labels_res)
df_feat_res = tm_res.add_feat_importance(df_feat=df_feat_res)
df_feat_res[["feature", "mean_dif", "abs_auc", "feat_importance"]].head()

,feature,mean_dif,abs_auc,feat_importance
0,"TMD-Pattern(C,6,7,8)-VENT840101",-0.467,0.5,7.619
1,"TMD-Pattern(C,6,7,8)-RACS820103",0.446,0.5,6.586
2,"TMD-Pattern(C,4,5)-WILM950102",-0.444,0.5,3.000
3,"TMD-Pattern(N,5,6)-WILM950102",-0.444,0.5,3.548
4,"TMD-Segment(3,4)-WILM950102",-0.444,0.5,5.014


In [6]:
# Domain level: splits sized to the named parts (positional + compositional)
split_kws_dom = sf.get_split_kws(n_split_min=1, n_split_max=10, steps_pattern=[3, 4],
                                 n_min=2, n_max=4, len_max=10)
cpp_dom = aa.CPP(df_parts=df_parts_dom, split_kws=split_kws_dom)
df_feat_dom = cpp_dom.run(labels=labels_dom, n_filter=20, n_jobs=1)  # n_jobs=1 avoids a spawn issue on Python 3.14/macOS

X_dom = sf.feature_matrix(features=df_feat_dom["feature"], df_parts=df_parts_dom)
tm_dom = aa.TreeModel().fit(X_dom, labels=labels_dom)
df_feat_dom = tm_dom.add_feat_importance(df_feat=df_feat_dom)
df_feat_dom[["feature", "mean_dif", "abs_auc", "feat_importance"]].head()

,feature,mean_dif,abs_auc,feat_importance
0,"TMD-Pattern(C,1,4)-VENT840101",0.700,0.5,7.786
1,"TMD-Pattern(C,4,8)-ROBB760110",-0.646,0.5,5.167
2,"TMD-Pattern(C,4,8)-COHE430101",0.627,0.5,4.333
3,"TMD-Pattern(C,1,4,8)-CHAM830104",0.600,0.5,4.529
4,"TMD-Pattern(C,4,8)-NAKH900112",0.570,0.5,9.708


In [7]:
# Protein level: compositional Segment(1,1) over the whole chain
split_kws_prot = sf.get_split_kws(split_types="Segment", n_split_min=1, n_split_max=1)
cpp_prot = aa.CPP(df_parts=df_parts_prot, split_kws=split_kws_prot)
df_feat_prot = cpp_prot.run(labels=labels_prot, n_filter=15, n_jobs=1)  # n_jobs=1 avoids a spawn issue on Python 3.14/macOS

X_prot = sf.feature_matrix(features=df_feat_prot["feature"], df_parts=df_parts_prot)
tm_prot = aa.TreeModel().fit(X_prot, labels=labels_prot)
df_feat_prot = tm_prot.add_feat_importance(df_feat=df_feat_prot)
df_feat_prot[["feature", "mean_dif", "abs_auc", "feat_importance"]].head()

,feature,mean_dif,abs_auc,feat_importance
0,"TMD-Segment(1,1)-KARS160110",-0.311,0.5,9.763
1,"TMD-Segment(1,1)-VASM830101",0.257,0.5,10.987
2,"TMD-Segment(1,1)-PALJ810111",0.254,0.5,10.880
3,"TMD-Segment(1,1)-MAXF760103",0.249,0.5,8.125
4,"TMD-Segment(1,1)-NAKH920101",-0.216,0.5,7.269


## Output

Each level returns its own `df_feat` **signature** (one row per selected feature, `feature` = `Part-Split-Scale`), but the feature ids read differently because the parts differ:

| Level | `df_parts` columns | Example feature | Reads as |
| --- | --- | --- | --- |
| residue | `tmd` (the window) | `TMD-Pattern(C,6,7,8)-...` | which **positions** in the window, which property |
| domain | `jmd_n`, `tmd`, `jmd_c` | `TMD-Pattern(C,1,4)-...` | which **sub-region** of the domain, which property |
| protein | `tmd` (whole chain) | `TMD-Segment(1,1)-...` | a **whole-chain composition** property (no position) |

The downstream columns are identical across levels: `mean_dif` (signed test - reference difference), `abs_auc` (separation strength), and `feat_importance` (tree-based rank). Visualise any of them with `aa.CPPPlot().feature_map(df_feat=...)` exactly as in Protocol 1 - the plot needs the `feat_importance` column added above.

## How to interpret

| Output | Non-expert reading |
| --- | --- |
| high `abs_auc` | strong group-separating property |
| positive `mean_dif` | property is **higher** in the test class |
| negative `mean_dif` | property is **higher** in the reference class |
| a `Segment(1,1)` feature (protein level) | a **compositional** signal - it does not depend on position |
| a `Pattern(...)` / multi-segment feature (residue/domain) | a **positional** signal - it depends on *where* in the part |

The level constrains the *kind* of feature you can read:

- **Residue** signatures are purely **positional** - e.g. CASPASE3 cleavage depends on which flanking position (P2, P3, P4...) carries which property around the scissile bond.
- **Protein** signatures are purely **compositional** - amyloid aggregation depends on overall composition, not position, so only `Segment(1,1)` features appear.
- **Domain** signatures mix both - the gamma-secretase TMD shows positional sub-region effects *and* whole-part composition.

As always, read the *signature* (coherent blocks of related features), not a single cell.

## Common mistakes

- **Calling `CPP(df_seq=...)`** - `CPP` takes `df_parts`; always build them with `SequenceFeature.get_df_parts` first (true at every level).
- **Forgetting that "part" means different things per level** - a window (residue), a named sub-region (domain), or the whole chain (protein). Name and interpret it accordingly.
- **Using positional splits at the protein level** - whole-chain composition is best captured by the **compositional** `Segment(1,1)`; multi-segment / `Pattern` splits over-fragment a position-agnostic signal.
- **Letting `split_kws` exceed the shortest part** - segments and patterns cannot be longer than the smallest part (e.g. a 10-residue JMD or a 9-residue window), or `CPP` raises a "too short sequences" error. Size `n_split_max` / `len_max` to the parts.
- **Confusing residue with domain** - both can use a single `tmd` part, but residue windows are short, fixed-length, and anchored at a site, while domain parts come from `tmd_start`/`tmd_stop` and carry flanking JMDs (`jmd_n_len`/`jmd_c_len`).

## Next step

- **Construct the comparison set** - for residue-level tasks you must first sample reference (non-site / non-cleaved) windows: see *Protocol 3 - Construct sets / sampling* (`AAWindowSampler`).
- **Read the full signature** - turn any level's `df_feat` into a feature map: see *Protocol 1 - CPP signature*.
- **Trust the result** - repeated-CV, bootstrap CIs and shuffled-label controls: see *Protocol 4 - Validate / Can I trust this?*
- **Engineer non-sequence features** - embeddings / structure / annotations via `CPP.run_num`: see the *Engineer features* protocol.